In [1]:
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup as bs
import random
import time
import json
import os
import re

In [2]:
# 測試連線

import requests

url = 'https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/'
# 加入verify=False 「叫 Python 忽略安全憑證檢查」。
response = requests.get(url, verify=False)

print(response)
# print(response.text)

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tisvcloud.freeway.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


<Response [200]>


In [ ]:
# 階段一：先爬簡單的，單一路徑

import requests

# 直接指定
url = "https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/20260518/23/TDCS_M03A_20260518_235500.csv"

response = requests.get(url, verify=False)
print(response)

with open("20260518_235500.csv", "wb") as f:
    f.write(response.content)

print("下載完成")

In [4]:
import time
import random
import requests
from datetime import datetime, timedelta

# 設定起止日期
start_date = datetime(2026, 1, 26)
end_date = datetime(2026, 1, 28)

base_url = "https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A_{}.tar.gz"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

current_date = start_date
while current_date <= end_date:
    # 格式化日期字串 (YYYYMMDD)
    date_str = current_date.strftime("%Y%m%d")
    file_url = base_url.format(date_str)
    
    # 發起請求下載檔案
    res = requests.get(file_url, headers=headers)
    if res.status_code == 200:
        with open(f"M03A_{date_str}.tar.gz", "wb") as f:
            f.write(res.content)
        print(f"✅ 成功下載：M03A_{date_str}.tar.gz")
    else:
        print(f"❌ 下載失敗 {date_str}，狀態碼：{res.status_code}")
    
    # 增加微小亂數間隔
    time.sleep(random.uniform(1.0, 2.5))
    
    # 日期加一天
    current_date += timedelta(days=1)

SSLError: HTTPSConnectionPool(host='tisvcloud.freeway.gov.tw', port=443): Max retries exceeded with url: /history/TDCS/M03A/M03A_20260126.tar.gz (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Missing Subject Key Identifier (_ssl.c:1028)')))

In [5]:
import time
import random
import requests
from datetime import datetime, timedelta

# 1. 關閉 SSL 驗證警告訊息 (可加可不加，加上去畫面比較乾淨)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 設定起止日期
start_date = datetime(2026, 1, 26)
end_date = datetime(2026, 1, 28)

base_url = "https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A_{}.tar.gz"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")
    file_url = base_url.format(date_str)
    
    try:
        # 2. 關鍵修正：加上 verify=False 跳過 SSL 憑證檢查
        res = requests.get(file_url, headers=headers, verify=False, timeout=15)
        
        if res.status_code == 200:
            with open(f"M03A_{date_str}.tar.gz", "wb") as f:
                f.write(res.content)
            print(f"✅ 成功下載：M03A_{date_str}.tar.gz")
        else:
            print(f"❌ 下載失敗 {date_str}，狀態碼：{res.status_code}")
            
    except Exception as e:
        print(f"⚠️ 連線發生例外狀況: {e}")
    
    # 增加微小亂數間隔
    time.sleep(random.uniform(1.0, 2.5))
    current_date += timedelta(days=1)

✅ 成功下載：M03A_20260126.tar.gz
✅ 成功下載：M03A_20260127.tar.gz
✅ 成功下載：M03A_20260128.tar.gz


In [3]:
from datetime import datetime, timedelta

# 1. 想知道 timedelta 到底有哪些屬性與方法？
print(dir(timedelta))

# 2. 想知道 datetime.now() 的具體用法與參數？
help(datetime.now)

['__abs__', '__add__', '__bool__', '__class__', '__delattr__', '__dir__', '__divmod__', '__doc__', '__eq__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__mod__', '__mul__', '__ne__', '__neg__', '__new__', '__pos__', '__radd__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmod__', '__rmul__', '__rsub__', '__rtruediv__', '__setattr__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__truediv__', 'days', 'max', 'microseconds', 'min', 'resolution', 'seconds', 'total_seconds']
Help on built-in function now:

now(tz=None) class method of datetime.datetime
    Returns new datetime object representing current time local to tz.

      tz
        Timezone object.

    If no tz is specified, uses local timezone.



In [11]:
from datetime import datetime, timedelta

# --- 驗證 help(datetime.now) ---
# 因為 tz 預設是 None，所以不帶參數就會拿到「本地現在時間」
now = datetime.now()
print("現在時間：", now)


# --- 驗證 dir(timedelta) 抓出來的屬性 ---
# 假設我們建立一個 2 天又 3 小時（10800 秒）的時間差
delta = timedelta(days=2, hours=3)

# 抓取我們剛剛在 dir() 看到的關鍵屬性
print("天數部分 (days):", delta.days)             # 印出: 2
print("秒數部分 (seconds):", delta.seconds)       # 印出: 10800
print("全部換算成總秒數 (total_seconds()):", delta.total_seconds()) # 印出: 183600.0

現在時間： 2026-07-25 22:21:48.966078
天數部分 (days): 2
秒數部分 (seconds): 10800
全部換算成總秒數 (total_seconds()): 183600.0
